<a href="https://colab.research.google.com/github/chawlaniharika23/Fraud_Detection_Research/blob/main/notebooks/05_improved_logistic_stacking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 05 — Leakage-Aware Logistic Stacking

## Financial Fraud Detection: IEEE-CIS

### Objective
Combine the six base classifiers trained in Notebook 04 using a logistic-regression meta-learner. Fit the meta-learner on an earlier portion of the chronological validation period and select its operating threshold on a later, disjoint portion.

### Important terminology
The supplied baseline notebook calls this experiment **Dynamic Stacking**. The implemented method is actually a **static probability-level logistic stacking ensemble**: it does not update weights online or dynamically over time. In the research paper, describe the method as logistic stacking unless a genuinely time-adaptive weighting mechanism is implemented and tested.

### Improvements
- Uses all six models from Notebook 04, including SVM and MLP.
- Uses a chronological, disjoint meta-fit / threshold-selection split within validation data.
- Fits the meta-learner only on base-model predictions from the earlier validation segment.
- Selects the decision threshold on the later validation segment only.
- Reports PR-AUC, ROC-AUC, F1, precision, recall, MCC, balanced accuracy, FPR and Brier score.
- Saves artifacts using the filenames expected by Notebook 06.
- Keeps the final test partition completely untouched.

## 1. Imports and reproducibility

In [1]:
import os
import gc
import json
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn

from scipy.sparse import load_npz
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix,
    brier_score_loss,
    precision_recall_curve
)
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
META_FIT_FRACTION = 0.70
MIN_FRAUD_CASES_PER_PARTITION = 10

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

NumPy: 2.4.6
Pandas: 2.2.3
scikit-learn: 1.6.1


## 2. Mount Google Drive and configure paths

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
BASE_DIR = Path("/content/drive/MyDrive/fraud-detection-research")
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RECOVERY_DIR = PROCESSED_DIR / "recovered"

BASE_MODEL_DIR = BASE_DIR / "results" / "base_models"
STACKING_DIR = BASE_DIR / "results" / "dynamic_stacking"
PREDICTION_DIR = BASE_MODEL_DIR / "validation_predictions"

STACKING_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

VAL_MATRIX_PATH = RECOVERY_DIR / "ieee_X_validation_selected_100.npz"
VAL_TARGET_PATH = PROCESSED_DIR / "ieee_y_validation.parquet"

MODEL_NAMES = ["LR", "DT", "RF", "GBT", "SVM", "MLP"]

required_paths = [VAL_MATRIX_PATH, VAL_TARGET_PATH]
required_paths += [BASE_MODEL_DIR / f"{name}_model.joblib" for name in MODEL_NAMES]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Required Notebook 03/04 artifacts are missing:\n"
        + "\n".join(missing)
        + "\nComplete Notebooks 03 and 04 before running Notebook 05."
    )

print("Stacking output directory:", STACKING_DIR)

Stacking output directory: /content/drive/MyDrive/fraud-detection-research/results/dynamic_stacking


## 3. Load validation data and base models

In [4]:
X_val = load_npz(VAL_MATRIX_PATH).tocsr().astype(np.float32)
y_val = pd.read_parquet(VAL_TARGET_PATH)["isFraud"].to_numpy(dtype=np.int8)

if X_val.shape[0] != len(y_val):
    raise ValueError("Validation features and labels are not aligned.")
if X_val.shape[1] != 100:
    raise ValueError(f"Expected 100 selected features; found {X_val.shape[1]}.")
if not np.isin(y_val, [0, 1]).all():
    raise ValueError("Validation labels must be binary.")

base_models = {
    name: joblib.load(BASE_MODEL_DIR / f"{name}_model.joblib")
    for name in MODEL_NAMES
}

print("Validation matrix:", X_val.shape)
print("Validation fraud count:", int(y_val.sum()))
print("Validation fraud rate:", f"{y_val.mean() * 100:.4f}%")
print("Loaded base models:", list(base_models))

Validation matrix: (88581, 100)
Validation fraud count: 3042
Validation fraud rate: 3.4341%
Loaded base models: ['LR', 'DT', 'RF', 'GBT', 'SVM', 'MLP']


## 4. Generate base-model validation probabilities

Each base model was trained in Notebook 04 using the earlier training partition. Here, each fitted model generates a fraud probability for every validation observation. These predictions form the meta-feature matrix.

The base models are not refitted on validation data.

In [5]:
prediction_columns = {}

for name, model in base_models.items():
    if not hasattr(model, "predict_proba"):
        raise TypeError(f"{name} does not expose predict_proba().")

    probabilities = model.predict_proba(X_val)[:, 1]
    probabilities = np.asarray(probabilities, dtype=np.float32)

    if len(probabilities) != len(y_val):
        raise ValueError(f"{name} produced a misaligned prediction vector.")
    if not np.isfinite(probabilities).all():
        raise ValueError(f"{name} produced non-finite probabilities.")
    if ((probabilities < 0) | (probabilities > 1)).any():
        raise ValueError(f"{name} produced values outside [0, 1].")

    prediction_columns[name] = probabilities
    print(f"{name}: validation probabilities generated.")

prediction_df = pd.DataFrame(prediction_columns)
P = prediction_df[MODEL_NAMES].to_numpy(dtype=np.float64)

print("Meta-feature matrix:", P.shape)
display(prediction_df.head())

LR: validation probabilities generated.
DT: validation probabilities generated.
RF: validation probabilities generated.
GBT: validation probabilities generated.
SVM: validation probabilities generated.
MLP: validation probabilities generated.
Meta-feature matrix: (88581, 6)


,LR,DT,RF,GBT,SVM,MLP
0,0.336899,0.229481,0.231805,0.292375,0.024811,0.006131
1,0.977924,0.988783,0.871012,0.792043,0.347192,0.989150
2,0.501668,0.697909,0.662178,0.604121,0.032786,0.004963
3,0.501977,0.229481,0.237118,0.283114,0.038490,0.006142
4,0.357333,0.229481,0.184212,0.268839,0.025891,0.003581


## 5. Save base predictions for reproducibility

The probability matrix is saved in the same model order used by the meta-learner. This makes the stacking experiment reproducible and avoids ambiguity about column order.

In [6]:
np.savez_compressed(
    PREDICTION_DIR / "base_model_validation_probabilities.npz",
    **{
        f"{name}_validation_probability": prediction_df[name].to_numpy(dtype=np.float32)
        for name in MODEL_NAMES
    }
)
np.save(PREDICTION_DIR / "validation_labels.npy", y_val)

with open(BASE_MODEL_DIR / "base_model_order.json", "w") as file:
    json.dump(MODEL_NAMES, file, indent=2)

print("Saved base-model validation probabilities and labels.")

Saved base-model validation probabilities and labels.


## 6. Chronologically separate meta-fitting and threshold selection

The validation period is kept in its existing chronological order. Its earlier 70% is used to fit the meta-learner; its later 30% is reserved for selecting the classification threshold. This avoids randomly mixing later validation observations into the meta-fitting subset.

This is a holdout strategy within validation, not out-of-fold stacking. The final test partition remains untouched.

In [7]:
n_validation = len(y_val)
meta_fit_end = int(np.floor(n_validation * META_FIT_FRACTION))

meta_fit_idx = np.arange(0, meta_fit_end)
threshold_idx = np.arange(meta_fit_end, n_validation)

P_meta = P[meta_fit_idx]
y_meta = y_val[meta_fit_idx]

P_threshold = P[threshold_idx]
y_threshold = y_val[threshold_idx]

if len(np.unique(y_meta)) < 2 or len(np.unique(y_threshold)) < 2:
    raise ValueError(
        "Both chronological validation subsets must contain legitimate and fraud cases. "
        "Inspect the split or choose a larger validation period."
    )

if int(y_meta.sum()) < MIN_FRAUD_CASES_PER_PARTITION:
    raise ValueError("Too few fraud cases in the meta-fitting subset.")
if int(y_threshold.sum()) < MIN_FRAUD_CASES_PER_PARTITION:
    raise ValueError("Too few fraud cases in the threshold-selection subset.")

print("Meta-fit rows:", len(meta_fit_idx))
print("Meta-fit time/order range: first validation segment")
print("Meta-fit fraud cases:", int(y_meta.sum()))
print("Threshold-selection rows:", len(threshold_idx))
print("Threshold-selection fraud cases:", int(y_threshold.sum()))

Meta-fit rows: 62006
Meta-fit time/order range: first validation segment
Meta-fit fraud cases: 2196
Threshold-selection rows: 26575
Threshold-selection fraud cases: 846


## 7. Fit the logistic-regression meta-learner

The meta-learner receives six probability values per transaction, one from each base classifier. L2 regularization limits extreme coefficients and helps control overfitting in the small meta-feature space.

No threshold-selection labels are used to fit this model.

In [8]:
meta_model = LogisticRegression(
    C=1.0,
    penalty="l2",
    solver="lbfgs",
    max_iter=2000,
    random_state=RANDOM_STATE
)

meta_fit_start = time.time()
meta_model.fit(P_meta, y_meta)
meta_fit_seconds = time.time() - meta_fit_start

threshold_probabilities = meta_model.predict_proba(P_threshold)[:, 1]

if not np.isfinite(threshold_probabilities).all():
    raise ValueError("Meta-learner produced non-finite probabilities.")

print("Meta-learner fitted in:", f"{meta_fit_seconds:.3f} seconds")
print("Meta-learner coefficients:")
display(pd.DataFrame({
    "Base_Model": MODEL_NAMES,
    "Coefficient": meta_model.coef_[0]
}).sort_values("Coefficient", ascending=False))

Meta-learner fitted in: 0.121 seconds
Meta-learner coefficients:


,Base_Model,Coefficient
2,RF,5.011523
5,MLP,2.993833
3,GBT,0.805702
1,DT,0.346138
0,LR,0.037289
4,SVM,-0.154058


## 8. Select the operating threshold on the reserved validation segment

The threshold is selected by maximizing validation F1 on the later, disjoint validation segment. The threshold is then frozen and passed to Notebook 06 for one-time evaluation on the untouched test partition.

If deployment has a specific false-positive budget or minimum-recall requirement, that operational objective should replace F1 maximization and be defined before final testing.

In [9]:
precision, recall, thresholds = precision_recall_curve(
    y_threshold,
    threshold_probabilities
)

if len(thresholds) == 0:
    raise ValueError("Threshold selection failed: no candidate thresholds returned.")

f1_values = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-12)
)

best_idx = int(np.nanargmax(f1_values))
selected_threshold = float(thresholds[best_idx])
selected_validation_f1 = float(f1_values[best_idx])

threshold_predictions = (
    threshold_probabilities >= selected_threshold
).astype(np.int8)

tn, fp, fn, tp = confusion_matrix(
    y_threshold, threshold_predictions, labels=[0, 1]
).ravel()

threshold_selection_metrics = {
    "Threshold": selected_threshold,
    "Precision": precision_score(y_threshold, threshold_predictions, zero_division=0),
    "Recall": recall_score(y_threshold, threshold_predictions, zero_division=0),
    "F1": f1_score(y_threshold, threshold_predictions, zero_division=0),
    "PR_AUC": average_precision_score(y_threshold, threshold_probabilities),
    "ROC_AUC": roc_auc_score(y_threshold, threshold_probabilities),
    "MCC": matthews_corrcoef(y_threshold, threshold_predictions),
    "FPR": fp / (fp + tn) if (fp + tn) else 0.0,
    "Brier_Score": brier_score_loss(y_threshold, threshold_probabilities),
    "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)
}

print("Selected threshold:", f"{selected_threshold:.8f}")
display(pd.DataFrame([threshold_selection_metrics]).round(6))

Selected threshold: 0.22881593


,Threshold,Precision,Recall,F1,PR_AUC,ROC_AUC,MCC,FPR,Brier_Score,TN,FP,FN,TP
0,0.228816,0.440922,0.361702,0.397403,0.391567,0.85894,0.381577,0.01508,0.024151,25341,388,540,306


## 9. Compare the stack with base models on the threshold-selection subset

In [10]:
comparison_rows = []

for model_name in MODEL_NAMES:
    base_probability = prediction_df[model_name].to_numpy()[threshold_idx]
    base_prediction = (base_probability >= 0.5).astype(np.int8)

    comparison_rows.append({
        "Model": model_name,
        "Threshold": 0.50,
        "Precision": precision_score(y_threshold, base_prediction, zero_division=0),
        "Recall": recall_score(y_threshold, base_prediction, zero_division=0),
        "F1": f1_score(y_threshold, base_prediction, zero_division=0),
        "PR_AUC": average_precision_score(y_threshold, base_probability),
        "ROC_AUC": roc_auc_score(y_threshold, base_probability)
    })

comparison_rows.append({
    "Model": "Logistic_Stacking",
    "Threshold": selected_threshold,
    "Precision": threshold_selection_metrics["Precision"],
    "Recall": threshold_selection_metrics["Recall"],
    "F1": threshold_selection_metrics["F1"],
    "PR_AUC": threshold_selection_metrics["PR_AUC"],
    "ROC_AUC": threshold_selection_metrics["ROC_AUC"]
})

stacking_comparison = pd.DataFrame(comparison_rows)
stacking_comparison.to_csv(
    STACKING_DIR / "validation_stacking_comparison.csv",
    index=False
)
display(stacking_comparison.sort_values("PR_AUC", ascending=False).round(5))

,Model,Threshold,Precision,Recall,F1,PR_AUC,ROC_AUC
6,Logistic_Stacking,0.22882,0.44092,0.36170,0.39740,0.39157,0.85894
2,RF,0.50000,0.20220,0.50000,0.28795,0.36447,0.84269
5,MLP,0.50000,0.51835,0.26714,0.35257,0.33466,0.81953
3,GBT,0.50000,0.11909,0.70567,0.20379,0.32842,0.84377
0,LR,0.50000,0.07140,0.76123,0.13055,0.30115,0.80904
4,SVM,0.50000,0.80000,0.07092,0.13029,0.28801,0.80375
1,DT,0.50000,0.13118,0.56147,0.21267,0.26922,0.72602


## 10. Save model and downstream-compatible configuration

The configuration explicitly records that this is static logistic stacking, not an online dynamic-weighting algorithm.

In [11]:
joblib.dump(meta_model, STACKING_DIR / "meta_model.joblib", compress=3)

stacking_config = {
    "model_names": MODEL_NAMES,
    "threshold": selected_threshold,
    "method": "static probability-level logistic stacking",
    "meta_learner": "LogisticRegression",
    "meta_fit_strategy": "earlier 70 percent of chronological validation predictions",
    "threshold_strategy": "maximize F1 on later 30 percent of chronological validation predictions",
    "meta_fit_fraction": META_FIT_FRACTION,
    "random_state": RANDOM_STATE,
    "validation_order_preserved": True,
    "test_set_used": False
}
joblib.dump(stacking_config, STACKING_DIR / "stacking_config.joblib")

print("Saved meta-model and downstream-compatible stacking configuration.")

Saved meta-model and downstream-compatible stacking configuration.


## 11. Save experiment summary and validation diagnostics

In [12]:
summary = {
    "experiment": "IEEE-CIS chronological holdout logistic stacking",
    "ensemble_name_for_legacy_output_compatibility": "Dynamic_Stacking",
    "actual_method": "Static probability-level logistic stacking",
    "base_models": MODEL_NAMES,
    "meta_learner": "LogisticRegression",
    "validation_rows": int(n_validation),
    "meta_fit_rows": int(len(meta_fit_idx)),
    "threshold_selection_rows": int(len(threshold_idx)),
    "meta_fit_fraud_cases": int(y_meta.sum()),
    "threshold_selection_fraud_cases": int(y_threshold.sum()),
    "threshold": selected_threshold,
    "threshold_selection_metrics": threshold_selection_metrics,
    "meta_fit_seconds": meta_fit_seconds,
    "limitations": [
        "This is static logistic stacking, not an online or dynamically updated ensemble.",
        "Meta-learner is fitted on an earlier validation segment rather than out-of-fold training predictions.",
        "The threshold is selected on a later, disjoint validation segment.",
        "The independent test set is reserved for Notebook 06 and is not used here."
    ]
}
with open(STACKING_DIR / "stacking_experiment_summary.json", "w") as file:
    json.dump(summary, file, indent=2)

with open(STACKING_DIR / "stacking_validation_metrics.json", "w") as file:
    json.dump(threshold_selection_metrics, file, indent=2)

print("Saved experiment summary and validation diagnostics.")

Saved experiment summary and validation diagnostics.


## 12. Compatibility support


In [13]:
mlp_model_path = BASE_MODEL_DIR / "MLP_model.joblib"
legacy_nn_path = BASE_MODEL_DIR / "NN_model.joblib"

if not mlp_model_path.exists():
    raise FileNotFoundError(
        "MLP_model.joblib is missing. Run the revised Notebook 04 first."
    )

joblib.dump(joblib.load(mlp_model_path), legacy_nn_path, compress=3)
print("Created Notebook 06 compatibility alias:", legacy_nn_path.name)

Created Notebook 06 compatibility alias: NN_model.joblib


## 13. Final artifact checks

In [14]:
required_outputs = [
    STACKING_DIR / "meta_model.joblib",
    STACKING_DIR / "stacking_config.joblib",
    STACKING_DIR / "stacking_experiment_summary.json",
    STACKING_DIR / "stacking_validation_metrics.json",
    STACKING_DIR / "validation_stacking_comparison.csv",
    BASE_MODEL_DIR / "NN_model.joblib"
]

missing_outputs = [str(path) for path in required_outputs if not path.exists()]
if missing_outputs:
    raise FileNotFoundError("Missing output artifacts:\n" + "\n".join(missing_outputs))

# Reload the saved configuration to confirm the exact order and threshold.
saved_config = joblib.load(STACKING_DIR / "stacking_config.joblib")
assert saved_config["model_names"] == MODEL_NAMES
assert 0.0 <= saved_config["threshold"] <= 1.0
assert saved_config["test_set_used"] is False

print("All Notebook 05 artifacts verified.")
print("Base model order:", saved_config["model_names"])
print("Validation-selected threshold:", saved_config["threshold"])

All Notebook 05 artifacts verified.
Base model order: ['LR', 'DT', 'RF', 'GBT', 'SVM', 'MLP']
Validation-selected threshold: 0.2288159306842698
